# 04 - M4: fusion + concept bank + affinity (real CLIP)
**Runtime > T4 GPU.** Gate: `X` is (B,576,384); `alpha` is (B,576,16) with rows summing to 1; gradients reach both projections (and, end to end, the LoRA tensors).
Also records affinity diagnostics (entropy, concept usage) so we know how informative alpha is at init. The concept text table is encoded once and committed to `assets/concepts_text_768.npy`.

In [ ]:
# BOOTSTRAP - identical first cell in every notebook. Safe to re-run: clones the repo if missing, otherwise pulls.
import os, subprocess
from google.colab import drive, userdata
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/csm-clip'
os.makedirs(DRIVE_ROOT + '/runs', exist_ok=True)
GH_USER = 'bhargav-reddy-nasappagari'
try:
    tok = userdata.get('GITHUB_TOKEN')      # only needed if the repo is private
except Exception:
    tok = None
url = f"https://{tok + '@' if tok else ''}github.com/{GH_USER}/csm-clip.git"
if not os.path.exists('/content/csm-clip'):
    subprocess.run(['git', 'clone', '-q', url, '/content/csm-clip'], check=True)
else:
    subprocess.run(['git', '-C', '/content/csm-clip', 'pull', '-q', '--rebase', url, 'master'], check=True)
os.chdir('/content/csm-clip')
subprocess.run(['git', 'remote', 'set-url', 'origin', f'https://github.com/{GH_USER}/csm-clip.git'], check=True)  # keep token out of .git/config
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m pytest -q

In [ ]:
import numpy as np, torch, json
from datasets import load_dataset
from transformers import CLIPVisionModel, CLIPTextModelWithProjection, CLIPTokenizer
from src.data import load_manifest, ManifestDataset
from src.backbone import ClipLoRABackbone
from src import concepts as K
from src.fusion import ConceptFrontEnd, affinity_stats

# 1) concept text table: encode once with the frozen CLIP text tower, cache as .npy (committed to the repo)
CKPT = K.TEXT_CHECKPOINT
ASSET = 'assets/concepts_text_768.npy'
tm = CLIPTextModelWithProjection.from_pretrained(CKPT, torch_dtype=torch.float32)
tok = CLIPTokenizer.from_pretrained(CKPT)
T = K.load_or_encode(ASSET, tm, tok, device='cuda')
assert T.shape == (16, 768), T.shape
print('concept table', tuple(T.shape))
for i, p in enumerate(K.prompts()): print(i, 'FAKE' if K.CONCEPT_POLARITY[i] else 'REAL', '|', p)
del tm

In [ ]:
# 2) real batch through backbone (M3) -> front end (M4)
m = load_manifest('Midjourney')
ds = load_dataset('TheKernel01/Tiny-GenImage', revision=m['revision'])
tr = ManifestDataset(m, 'train', ds['train'], ds['validation'], protocol='controlled', train=False)
x = torch.stack([tr[i][0] for i in range(8)]).cuda()
clip = CLIPVisionModel.from_pretrained(CKPT, torch_dtype=torch.float32).cuda().eval()
bb = ClipLoRABackbone(clip, taps=(16, 24), r=8, alpha=16.0).cuda()
fe = ConceptFrontEnd(T, d_in=1024, d=384, n_taps=2).cuda()
print({k: sum(p.numel() for p in v.parameters() if p.requires_grad) for k, v in {'frontend': fe, 'lora': bb}.items()})

In [ ]:
# GATE 1: shapes + alpha rows sum to 1 (fp16 autocast, as in training)
torch.cuda.reset_peak_memory_stats()
with torch.autocast('cuda', dtype=torch.float16):
    out = fe(bb(x))
X, alpha = out['X'], out['alpha']
assert X.shape == (8, 576, 384), X.shape
assert alpha.shape == (8, 576, 16), alpha.shape
assert alpha.dtype == torch.float32 and torch.isfinite(alpha).all() and torch.isfinite(X).all()
s = (alpha.sum(-1) - 1).abs().max().item(); print('max |row sum - 1|', s); assert s < 1e-4
print('GATE 1 OK | peak GPU', f'{torch.cuda.max_memory_allocated()/2**30:.2f} GiB')

In [ ]:
# GATE 2: gradients reach both projections, the mix/scale, and (end to end) the LoRA tensors
with torch.no_grad():                      # B=0 at init zeroes A's gradient; nudge B once for the check
    for n, p in bb.named_parameters():
        if 'lora_B' in n: p.add_(0.01 * torch.randn_like(p))
bb.zero_grad(set_to_none=True); fe.zero_grad(set_to_none=True)
with torch.autocast('cuda', dtype=torch.float16):
    out = fe(bb(x))
(out['alpha'] * torch.arange(16, device='cuda', dtype=torch.float32)).sum().backward()
g = lambda p: p.grad is not None and p.grad.abs().sum().item() > 0
assert g(fe.fusion.proj.weight), 'visual projection got no gradient'
assert g(fe.affinity.text_proj.weight), 'text projection got no gradient'
assert g(fe.fusion.mix_logits) and g(fe.affinity.log_scale)
assert fe.affinity.text_embeds.grad is None
assert all(g(p) for p in bb.lora_parameters()), 'LoRA did not receive gradient through the front end'
print('GATE 2 OK | mix weights', fe.fusion.mix_weights().tolist(), '| scale', fe.affinity.log_scale.exp().item())

In [ ]:
# Diagnostics (not a gate): how informative is alpha at init?
with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
    stats = affinity_stats(fe(bb(x))['alpha'])
print(json.dumps(stats, indent=1))

In [ ]:
# Save: results to Drive + repo runs/, and the concept table to Drive as a backup (repo copy is assets/)
import shutil
out = {'milestone': 'M4', 'shapes': {'X': [8, 576, 384], 'alpha': [8, 576, 16]},
       'row_sum_max_err': s, 'affinity_stats_init': stats, 'checkpoint': CKPT, 'gate': 'PASS'}
for p in (f'{DRIVE_ROOT}/runs/m4_gate.json', '/content/csm-clip/runs/m4_gate.json'):
    json.dump(out, open(p, 'w'), indent=1)
shutil.copy(ASSET, DRIVE_ROOT + '/concepts_text_768.npy')
print(out)
# Then: commit assets/concepts_text_768.npy + runs/m4_gate.json from your machine (or push from Colab).